# Python basics: values and types

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-01-values-and-types.ipynb)

Python is the language most data engineers write first. Before you can build pipelines you need to know what a value
is, how Python stores it, and which operations are safe. This chapter covers numbers, strings, booleans and `None`,
and the traps that cause wrong answers in real data: integer division, floating-point error and money.

Every example is runnable and every claim is asserted, so a wrong statement fails loudly.

## Values, names and types

A **value** is an object with a type. A **name** is a label that points at a value. Assigning a name does not copy
the value; it attaches the label.

`type()` tells you the type of a value. `isinstance()` is the check to use in code, because it also accepts subclasses.

### Where you'll use it

- **Schema checks:** before loading a file, check each field's type. A `"12.0"` string where an integer belongs is the most common reason a load fails at 3 a.m.
- **Logs and debugging:** `type(x)` and `repr(x)` in a log line tell you what the pipeline actually received, not what you expected.

In [ ]:
count = 42
price = 19.99
label = "orders"
is_active = True
nothing = None

assert type(count) is int
assert type(price) is float
assert type(label) is str
assert type(is_active) is bool
assert type(nothing) is type(None)

# bool is a subclass of int, so isinstance(True, int) is True.
assert isinstance(True, int)
assert isinstance(count, (int, float))  # check several types at once

## Numbers: integers, floats and division

- `int` has unlimited size. Python never overflows.
- `float` is a 64-bit binary floating-point number. Most decimal fractions are only approximated.
- `/` always returns a float. `//` is floor division (rounds toward negative infinity). `%` is the remainder.

Floor division on negative numbers surprises people. `-7 // 2` is `-4`, not `-3`, because it rounds down.

### Where you'll use it

- **Bucketing and paging:** `//` and `%` split a row number into a page and an offset, or an epoch in seconds into hour buckets.
- **Rates and averages:** `/` gives a float for a rate such as errors per request. Guard it with a zero check, because a day with no traffic must not crash the job.

In [ ]:
assert 7 / 2 == 3.5
assert 7 // 2 == 3
assert 7 % 2 == 1
assert -7 // 2 == -4          # floor, not truncation
assert -7 % 2 == 1            # the remainder keeps the sign of the divisor
assert 2 ** 100 > 10 ** 30    # no overflow
assert round(2.5) == 2        # round() rounds halves to the even neighbour
assert round(3.5) == 4

## The floating-point trap

`0.1` has no exact binary representation, so `0.1 + 0.2` is not exactly `0.3`. Never compare floats with `==`.
Use `math.isclose` for measurements, and use `Decimal` for money (next section).

### Where you'll use it

- **Reconciliation:** comparing a total from two systems. A sum that is off by `1e-12` should pass, so use a tolerance and report it.
- **Data quality rules:** a check such as "amount equals qty times price" needs `math.isclose`, or it fails on valid rows.

In [ ]:
import math

assert 0.1 + 0.2 != 0.3
assert math.isclose(0.1 + 0.2, 0.3)
assert f"{0.1 + 0.2:.20f}".startswith("0.3000000000000000444")

## Money: use `Decimal`

`Decimal` stores decimal digits exactly. Build it from a **string**. `Decimal(0.1)` captures the binary error of the
float, so the string form is the one that keeps the intended value.

### Where you'll use it

- **Finance and billing pipelines:** invoices, tax, currency conversion and ledger totals. Floats here cause mismatched totals that auditors find.
- **Warehouse loads:** when a `DECIMAL(12,2)` column is read into Python, `Decimal` keeps the same precision on the way out.

In [ ]:
from decimal import Decimal

assert Decimal("0.10") + Decimal("0.20") == Decimal("0.30")
assert Decimal(0.1) != Decimal("0.1")   # the float error comes along
assert str(Decimal("1234.50") * 2) == "2469.00"

## Strings

Strings are immutable: you cannot change one in place, you build a new string. Useful tools:

- **f-strings** format values: `f"{value:,}"` adds thousands separators, `f"{ratio:.1%}"` formats a percentage.
- **Methods** return new strings: `strip`, `lower`, `replace`, `split`, `join`, `startswith`.
- **Slices** take parts: `s[start:stop]`, and `s[::-1]` reverses.

### Where you'll use it

- **Cleaning raw text:** trimming whitespace, normalising case and splitting delimited fields before they reach a table.
- **Building keys and paths:** `"-".join` for partition names, `f"{x:05d}"` for zero-padded file sequence numbers.

In [ ]:
name = "  Ana Lopez  "
assert name.strip() == "Ana Lopez"
assert name.strip().lower() == "ana lopez"
assert "-".join(["2024", "01", "15"]) == "2024-01-15"
assert "2024-01-15".split("-") == ["2024", "01", "15"]

assert f"{1234567:,}" == "1,234,567"
assert f"{0.1234:.1%}" == "12.3%"
assert f"{42:05d}" == "00042"
assert "hello"[1:4] == "ell"
assert "hello"[::-1] == "olleh"

try:
    name[0] = "X"
except TypeError:
    pass  # strings cannot be changed in place
else:
    raise AssertionError("expected TypeError")

## Truthiness and `None`

In a condition, Python treats these values as false: `0`, `0.0`, `""`, `[]`, `{}`, `set()` and `None`. Everything else is
true. Data bugs often hide here: an empty string and a missing value behave the same in `if value:`, so check for
`None` explicitly when the difference matters.

### Where you'll use it

- **Null handling in ingestion:** a source sends `""`, `None` or a missing key. Treat each explicitly, because `if value:` lumps them together and loses a real zero.
- **Optional config:** `if setting is None` lets a configured `0` or empty string stay valid.

In [ ]:
def describe(value):
    if value is None:
        return "missing"
    if value == "":
        return "empty"
    return "present"

assert describe(None) == "missing"
assert describe("") == "empty"
assert describe("0") == "present"   # the string "0" is not empty
assert not 0 and not "" and not []

## Converting between types

`int()`, `float()` and `str()` convert when the text is valid. Invalid text raises `ValueError`. Conversion is where
raw files go wrong: `"12.0"` cannot go straight to `int`.

### Where you'll use it

- **Parsing source files:** CSV and JSON values arrive as text. Every pipeline converts them, and the conversion is where bad rows are found.
- **API responses:** IDs sent as strings must match integer keys in the warehouse before a join works.

In [ ]:
assert int("42") + 1 == 43
assert float("3.5") * 2 == 7.0
assert str(7) + "x" == "7x"
assert int(float("12.0")) == 12

try:
    int("12.0")
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

## Practice questions

1. Parse `"1,234.50"` into a `Decimal` and add `0.50`. Show the result as a string.
2. Write `mask_email(address)` that returns `"a***@example.com"` for `"ana@example.com"`: keep the first character of the
   local part and the whole domain.
3. Why is `bool("False")` true? Explain in one sentence, then assert it.

## Answers

### Question 1

In [ ]:
from decimal import Decimal

total = Decimal("1,234.50".replace(",", "")) + Decimal("0.50")
assert str(total) == "1235.00"

### Question 2

In [ ]:
def mask_email(address):
    local, domain = address.split("@", 1)
    return local[0] + "***@" + domain

assert mask_email("ana@example.com") == "a***@example.com"

### Question 3

`bool` only looks at whether a string is empty. Any non-empty string, including `"False"`, is true.

In [ ]:
assert bool("False") is True
assert bool("") is False

## Interview notes

- **`is` vs `==`.** `==` compares values; `is` compares identity. Use `is None`, and `==` for everything else.
- **Floats are approximate.** Compare with a tolerance, and keep money in `Decimal` or integer cents.
- **`/` always gives a float**, so `total / count` is a float even when both are ints. Use `//` when you need a floor.
- **Summary.** Know the type of every value you read from a file. Most pipeline bugs start as a string where a number was
  expected, or a missing value treated as empty.